# Prepare gene feature for matching analysis
For overlap with genes, we download the [Gencode gene set release 50 basic gene annotations](https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.basic.annotation.gtf.gz)

In [1]:
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve

In [2]:
supporting_data_dir = Path("supporting_data")
supporting_data_dir.mkdir(exist_ok=True)

gene_anno_url = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.basic.annotation.gtf.gz"
gene_anno_path = supporting_data_dir / "gencode.v50.basic.annotation.gtf.gz"

files = {
    gene_anno_url: gene_anno_path
}

for url, output_path in files.items():
    if output_path.exists():
        print(f"Already exists: {output_path}")
    else:
        print(f"Downloading {output_path.name}...")
        urlretrieve(url, output_path)

print("Done.")

Already exists: supporting_data/gencode.v50.basic.annotation.gtf.gz
Done.


In [3]:
# columns derived from https://www.gencodegenes.org/pages/data_format.html
gff_df = pd.read_csv(
    "supporting_data/gencode.v50.basic.annotation.gtf.gz",
    compression='gzip',
    delimiter='\t',
    header=4,
    names=[
        'chromosome', 'annotation_source', 'feature_type',
        'genomic_start', 'genomic_end',
        'score', 'genomic_strand', 'genomic_phase', 'additional_info'
    ]
)

In [4]:
gff_df = (
    gff_df
    .query("feature_type == 'gene'")
    .reset_index()
)

In [5]:
# extract additional_info fields; stored as a single string but contains key/value pairs

gff_df['info'] = (
    gff_df['additional_info']
    .str.split('; ') # split main string into key/value attribute strings
    .apply(lambda items: [item.split(' ') for item in items]) # split into key/value tuples
    .apply(dict)
)
info_df = pd.json_normalize(gff_df['info'])
for col in info_df.columns:
    info_df[col] = info_df[col].str.strip('\"') # remove quotation mark characters wrapped around values

In [6]:
gene_df = pd.concat(
    [
        gff_df,
        info_df
    ],
    axis=1
)

In [7]:
gene_df = gene_df.query("gene_type == 'protein_coding'")

In [8]:
gene_df['genomic_start'] = gene_df['genomic_start'].astype(int)
gene_df['genomic_end'] = gene_df['genomic_end'].astype(int)

In [9]:
gene_df = gene_df[['chromosome', 'genomic_start', 'genomic_end', 'gene_id', 'gene_name']]

In [10]:
gene_df['chromosome'] = gene_df['chromosome'].str.strip('chr')

In [11]:
gene_df.to_csv(
    'supporting_data/gene-feature-locations-preprocessed.csv',
    index=False
)